# Install libraries



In [1]:
# INSTALL
!pip install -q --no-warn-script-location langchain-groq langgraph tavily-python sympy matplotlib numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.9 MB/s eta 0:00:00


# Imports libraries

In [2]:
# IMPORTS
import os
import sympy as sp
from sympy import sympify, solve, diff, integrate, laplace_transform, Eq
from sympy.solvers.ode import classify_ode
import matplotlib.pyplot as plt
from io import BytesIO
import base64
import numpy as np
from scipy.optimize import minimize

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated, Sequence
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

**This code is responsible for securely managing API keys for Groq and Tavily in Google Colab. It prioritizes retrieving the keys from Colab Secrets for better security. If the keys are not available, it gracefully falls back to manual user input and then stores them in environment variables for use throughout the session.**

In [3]:
# API KEYS
print("API Key Setup")

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
except:
    GROQ_API_KEY = None
    TAVILY_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = input("Enter your Groq API Key: ").strip()
if not TAVILY_API_KEY:
    TAVILY_API_KEY = input("Enter your Tavily API Key: ").strip()

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

API Key Setup


**This is a custom AdvancedMathSolver class built using SymPy that can declare symbols, solve equations, and calculate derivatives with LaTeX support for mathematical expressions.**

In [4]:
# MATH SOLVER
class AdvancedMathSolver:
    def __init__(self):
        self.symbols = {}

    def declare_symbols(self, symbols_str: str):
        for s in symbols_str.split():
            self.symbols[s] = sp.Symbol(s)
        return f"Declared symbols: {list(self.symbols.keys())}"

    def solve_equation(self, eq_str: str, var: str = 'x'):
        try:
            eq = Eq(sympify(eq_str.split('=')[0]), sympify(eq_str.split('=')[1])) if '=' in eq_str else sympify(eq_str)
            variable = self.symbols.get(var, sp.Symbol(var))
            solutions = solve(eq, variable, dict=True)
            return {"result": solutions, "steps": f"Solved: {solutions}"}
        except Exception as e:
            return {"error": str(e)}

    def differentiate(self, expr: str, var: str = 'x', order: int = 1):
        try:
            expression = sympify(expr)
            variable = self.symbols.get(var, sp.Symbol(var))
            result = diff(expression, variable, order)
            return {"result": result, "latex": sp.latex(result)}
        except Exception as e:
            return {"error": str(e)}

**This code creates two important tools for my AI agent. The math_solver tool uses SymPy through a custom wrapper to solve equations and compute derivatives. The web_search tool integrates with Tavily API to allow the agent to perform real-time web searches and fetch current information. Both tools are decorated with @tool so they can be used by the LangGraph agent.**

In [5]:
# TOOLS
@tool
def math_solver(operation: str, params: dict = {}) -> str:
    """Advanced Math Tool: solve, differentiate, integrate, etc."""
    solver = AdvancedMathSolver()

    if "symbols" in params:
        solver.declare_symbols(params["symbols"])

    try:
        if operation == "solve":
            result = solver.solve_equation(params.get("equation", ""), params.get("var", "x"))
        elif operation == "diff":
            result = solver.differentiate(params.get("expr", ""), params.get("var", "x"), params.get("order", 1))
        else:
            return "Supported operations: solve, diff"

        if "error" in result:
            return f"Math Error: {result['error']}"
        return str(result.get("latex", result.get("result", "Done")))
    except Exception as e:
        return f"Tool Error: {str(e)}"


@tool
def web_search(query: str) -> str:
    """Search the web using Tavily"""
    try:
        from tavily import TavilyClient
        client = TavilyClient(api_key=TAVILY_API_KEY)
        response = client.search(query, max_results=5, search_depth="basic")
        return "\n".join([f"• {r['content'][:300]}..." for r in response.get('results', [])])
    except Exception as e:
        return f"Search Error: {str(e)}"

**This section sets up the Groq LLM (Llama-3.3-70B model) with proper error handling and connection testing. It also binds the custom tools (math_solver and web_search) to the LLM so that the agent can intelligently call these tools when needed.**

In [6]:
# LLM SETUP (with better error handling)
print("\n Connecting to Groq...")

try:
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.7,
        max_tokens=1500,
        api_key=GROQ_API_KEY
    )
    # Test connection
    test = llm.invoke("Say hello in one word")
    print(" Groq LLM Connected Successfully!")
except Exception as e:
    print(f" Groq Connection Failed: {e}")
    print("\n Solutions:")
    print("1. Go to https://console.groq.com/keys")
    print("2. Create a **new API key** (old one has expired)")
    print("3. Paste the new key when asked")
    raise SystemExit("Please restart with a valid Groq API Key")


tools = [math_solver, web_search]
llm_with_tools = llm.bind_tools(tools)


 Connecting to Groq...
 Groq LLM Connected Successfully!


**This code builds the LangGraph-based AI Agent. It creates a cyclic workflow between the LLM (agent node) and tools. The agent can reason, decide when to call tools (math solver or web search), execute them, and return to the agent for the final answer. Memory is maintained using MemorySaver so the agent has conversation context.**

In [7]:
# AGENT SETUP
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

def agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)

workflow = StateGraph(state_schema=AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

**This code creates the Gradio-based frontend for my AI Math Agent. It integrates the LangGraph agent with a clean chat UI that supports both Enter key and Send button. The interface maintains conversation memory, displays responses in real-time, and provides a user-friendly experience with Hinglish support. It also includes proper error handling and a clear chat functionality.**

In [8]:
# SYSTEM PROMPT
system_prompt = """
- intelligent math Agent.
- Friendly Hindi-English mix
- Math problems में step-by-step explanation
- Exact answer + numerical approximation
- Short, clear aur engaging
"""

# GLOBAL SETUP
config = {"configurable": {"thread_id": "kishor-math-agent"}}
messages = [SystemMessage(content=system_prompt)]

# CHAT FUNCTION
def chat_with_agent(user_input: str, history):
    if not user_input or not user_input.strip():
        return history, history

    messages.append(HumanMessage(content=user_input.strip()))

    try:
        response_content = ""
        for event in app.stream({"messages": messages}, config, stream_mode="values"):
            last_msg = event["messages"][-1]
            if isinstance(last_msg, AIMessage) and last_msg.content:
                response_content = last_msg.content

        messages.append(AIMessage(content=response_content))

        history = history + [(user_input, response_content)]
        return history, history

    except Exception as e:
        error_msg = f" Error: {str(e)}"
        history = history + [(user_input, error_msg)]
        return history, history


def clear_chat():
    global messages
    messages = [SystemMessage(content=system_prompt)]
    return [], []


# GRADIO INTERFACE
with gr.Blocks(title="Intelligent Math Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Intelligent Math Agent
    """)

    chatbot = gr.Chatbot(
        height=650,
        label="Chat History",
        bubble_full_width=False,
        show_copy_button=True
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="अपना सवाल लिखें और Enter दबाएँ...",
            label="Your Message",
            lines=1,                    # ← Important: Single Line
            max_lines=5,
            autofocus=True
        )

    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear Chat", variant="secondary")

    gr.Markdown("""
    **Note:** Enter Key दबाकर भेजें। अगर लंबा message लिखना हो तो Shift + Enter से नई लाइन डाल सकते हैं।
    """)

    # EVENTS
    # Enter Key Press
    msg.submit(
        fn=chat_with_agent,
        inputs=[msg, chatbot],
        outputs=[chatbot, chatbot]
    ).then(
        fn=lambda: "",
        inputs=None,
        outputs=msg
    )

    # Send Button
    send_btn.click(
        fn=chat_with_agent,
        inputs=[msg, chatbot],
        outputs=[chatbot, chatbot]
    ).then(
        fn=lambda: "",
        inputs=None,
        outputs=msg
    )

    clear_btn.click(
        fn=clear_chat,
        inputs=None,
        outputs=[chatbot]
    )

# LAUNCH
print("Launching Kishor Bhai ka Math Dost...")
demo.launch(
    share=True,
    debug=False,
    height=780
)

Launching Kishor Bhai ka Math Dost...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2505f3a42d75f2622a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
